In [ ]:
import time
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from arms import ARMS, FILTERS, FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.datasets import dataset_ref
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.sequence import all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights
from uavloc.track import experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "027-sequence-filters"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
K_MAX_HYPOTHESES = 5
PEAKS_PER_HYPOTHESIS = 3
PEAK_NMS_M = 10.0
MERGE_DISTANCE_M = 5.0
CHILD_FLOOR_SIGMA_M = 1.0
WINDOW_SIGMAS = 3.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_RUN = None                   # e.g. "023.3": score with that 023 correlation head instead of weighted NCC
HEAD_EXP = "023-learned-correlation-head"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages the 023 head never trained on
TUNING_DIR = "artifacts/tuning"
ARM_LIST = list(ARMS)
SCENARIO = "normal"
SPLIT = "test"
LOG_RUN = True
WAIT_FOR_RUNS = 0                 # log only once runs.jsonl holds this many runs, so ids follow the proposal
REGISTERED_KEY = "normal"         # normal | cold | head
REGISTERED_RUNS = {"normal": ["027.1", "027.2", "027.3", "027.4", "027.5"],
                   "cold": ["027.6", "027.7", "027.8", "027.9", "027.10"],
                   "head": ["027.11", "027.12", "027.13"]}
GATE = {"temperature": 0.25, "s_max_m": 3.0, "c_min": 0.01}   # the 007.8 values, gated arm only
TUNED = {"particles": {"temperature": 1.0, "q_pos_m_per_sqrt_s": 1.0},
         "hypotheses": {"temperature": 1.0, "q_pos_m_per_sqrt_s": 1.0}}   # from tuning, set per papermill run
BAR_FILTER = "particles"          # the filter with the lower tuning error, fixed before the test runs
NOTE = ""

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
tuning = pd.concat([pd.read_csv(p) for p in sorted(Path(TUNING_DIR).glob("shard-[0-9].csv"))], ignore_index=True)
len(specs), len(tuning)

# Model

In [ ]:
def filter_params(temperature: float, q_pos: float) -> FilterParams:
    return FilterParams(temperature, q_pos, N_PARTICLES, K_MAX_HYPOTHESES, PEAKS_PER_HYPOTHESIS, PEAK_NMS_M,
                        MERGE_DISTANCE_M, CHILD_FLOOR_SIGMA_M, WINDOW_SIGMAS, WINDOW_CAP_HALF_M, WINDOW_MARGIN_M)

In [ ]:
def correlator_for(map_raster):
    head = None if HEAD_RUN is None else HeadSpec(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt",
                                                  tuple(HEAD_ENCODER_CHANNELS), HEAD_REFINE_CHANNELS)
    return build_correlator(map_raster, CORRELATOR, DEVICE, head)

In [ ]:
def params_for(arm: str) -> FilterParams | None:
    return filter_params(TUNED[arm]["temperature"], TUNED[arm]["q_pos_m_per_sqrt_s"]) if arm in FILTERS else None

In [ ]:
factories = {arm: estimator_factory(arm, params_for(arm), GATE) for arm in ARM_LIST}
list(factories)

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M[SCENARIO], SIGMA0_M[SCENARIO], correlator_for, factories)

In [ ]:
tables = {arm: per_flight_table(found) for arm, found in outcomes.items()}
def arm_metrics(found) -> dict:
    clean = [o for o in found if o.spec.village in HEAD_CLEAN_VILLAGES]
    return {**flight_metrics(found), **{f"head_clean_{k}": v for k, v in flight_metrics(clean).items()}}

In [ ]:
metrics = {arm: arm_metrics(found) for arm, found in outcomes.items()}
pd.DataFrame(metrics).T

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
plot_end_error_cdf({arm: table["end_error_m"] for arm, table in tables.items()}, ax)
ax.set_title(f"027, {SCENARIO} start, {SPLIT} flights ({len(specs)})")
fig.tight_layout()

# Log

In [ ]:
def tuning_record(arm: str) -> dict | None:
    if arm not in FILTERS:
        return None
    rows = tuning[tuning["filter"] == arm]
    return {"grid": {"temperature": sorted(rows["temperature"].unique().tolist()),
                     "q_pos_m_per_sqrt_s": sorted(rows["q_pos_m_per_sqrt_s"].unique().tolist())},
            "selected_by": "mean_end_error_m on tuning villages, normal start",
            "chosen": TUNED[arm], "tuning_mean_end_error_m": float(rows["mean_end_error_m"].min())}

In [ ]:
def run_config(arm: str, registered: str) -> dict:
    return {"registered_run": registered, "arm": arm, "scenario": SCENARIO, "split": SPLIT,
            "route_set": dataset_ref(ROUTE_SET), "initial_error_m": INITIAL_ERROR_M[SCENARIO],
            "sigma0_m": SIGMA0_M[SCENARIO], "ins": asdict(set_config.ins), "correlator": CORRELATOR,
            "gate": GATE if arm == "gated" else None,
            "likelihood": "weighted-ncc" if HEAD_RUN is None else f"head {HEAD_RUN}",
            "filter": asdict(params_for(arm)) if arm in FILTERS else None,
            "tuning": tuning_record(arm), "bar_filter": BAR_FILTER}

In [ ]:
def log_arm(arm: str, registered: str) -> None:
    out = Path("artifacts")
    tables[arm].to_csv(out / f"flights-{registered}.csv", index=False)
    all_fixes(outcomes[arm]).to_csv(out / f"fixes-{registered}.csv", index=False)
    while len(load_runs(EXP)) < WAIT_FOR_RUNS:
        time.sleep(30)
    expected = f"027.{len(load_runs(EXP)) + 1}"
    if expected != registered:
        raise RuntimeError(f"run would log as {expected}, registered as {registered}")
    log_run(exp=EXP, config=run_config(arm, registered), metrics=metrics[arm], note=f"{registered}: {arm}, {SCENARIO} start. {NOTE}",
            figures=[fig], seed=SEED, device=DEVICE)

In [ ]:
if LOG_RUN:
    for arm, registered in zip(ARM_LIST, REGISTERED_RUNS[REGISTERED_KEY]):
        log_arm(arm, registered)